# Gray-Box AIA72 — Fallback-First Operational Policy v3 Freeze

Freezes the availability-preserving v3 policy after the Cycle-25 diagnostic failure of v2. Cycle 25 is now explicitly treated as development evidence. **Supplementary 2026 remains untouched and is reserved for clean future evaluation.**

In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

# ============================================================
# Gray-Box AIA72 — Operational Policy v3 Freeze
# Availability-preserving fallback-first redesign
# ============================================================
#
# Motivation:
# Cycle-25 diagnostic evaluation showed that v2 abstention disproportionately
# removed flare-producing cases. v3 therefore changes the *decision logic*:
#
# NORMAL   -> issue fixed SHARP+AIA fusion when fusion trust gates pass.
# DEGRADED -> otherwise, if SHARP is technically available, issue SHARP fallback.
# ABSTAIN  -> only when SHARP itself is technically unavailable.
#
# Important:
# - Cycle25 has been seen and is now development/diagnostic evidence.
# - No supplementary_2026 outcomes are read here.
# - No new numeric thresholds are tuned here.
# - q90 remains the primary normal-mode gate, inherited from the predeclared family.
# - q80/q95 remain sensitivity definitions only.
# - SHARP Mondrian state is retained as an advisory diagnostic, NOT a blocking gate.
#
# This notebook freezes v3 on the original conformal-calibration support only.

MASTER = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz")
MASTER_SHA = "17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

SHARP = Path("/home/abmoses2000/sharp72_predictions_frozen_20261002.csv.gz")
SHARP_SHA = "7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"

MOND = Path("/home/abmoses2000/graybox_aia72_conformal_mondrian_v2_20261005")
MOND_PROTOCOL = MOND / "protocol.json"
MOND_THRESHOLDS = MOND / "mondrian_thresholds.csv"

CUTS = Path("/home/abmoses2000/graybox_aia72_policy_freeze_v1_20261005/candidate_cutoffs.csv")

ALARM_DIR = Path("/home/abmoses2000/graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005")
ALARM_THRESHOLDS = ALARM_DIR / "alarm_thresholds.csv"

V2_POLICY = Path("/home/abmoses2000/graybox_aia72_policy_mondrian_v2_freeze_20261005")
V2_PROTOCOL = V2_POLICY / "protocol.json"

CYCLE25 = Path("/home/abmoses2000/graybox_aia72_cycle25_mondrian_v2_eval_20261005")
CYCLE25_SUMMARY = CYCLE25 / "summary.json"

OUT = Path("/home/abmoses2000/graybox_aia72_policy_v3_fallback_first_freeze_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

ROLE = "conformal_calibration"
EXPECTED_ROWS = 4168
PRIMARY_ALPHA = 0.10
SEEDS = [17,29,43]
PERCENTILES = [0.80,0.90,0.95]

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond,msg):
    if not cond:
        raise ValueError(msg)

def entropy_binary(p):
    p=np.asarray(p,float)
    q=np.clip(p,1e-12,1-1e-12)
    return -(q*np.log(q)+(1-q)*np.log(1-q))

def mondrian_state(p,q0,q1):
    p=np.asarray(p,float)
    include0=p<=q0
    include1=(1-p)<=q1
    out=np.empty(len(p),dtype=object)
    out[(~include0)&(~include1)]="empty"
    out[include0&(~include1)]="singleton_no_flare"
    out[(~include0)&include1]="singleton_flare"
    out[include0&include1]="doubleton"
    return out

for p in [MASTER,SHARP,MOND_PROTOCOL,MOND_THRESHOLDS,CUTS,ALARM_THRESHOLDS,V2_PROTOCOL,CYCLE25_SUMMARY]:
    require(p.is_file(), f"Missing frozen input: {p}")

require(sha256(MASTER)==MASTER_SHA,"Master SHA changed")
require(sha256(SHARP)==SHARP_SHA,"SHARP SHA changed")

v2_protocol=json.loads(V2_PROTOCOL.read_text())
cycle25_summary=json.loads(CYCLE25_SUMMARY.read_text())

require(v2_protocol["primary_policy_id"]=="mondrian_v2_q90","v2 primary changed")
require(cycle25_summary["role"]=="retrospective_cycle25","Cycle25 evidence mismatch")
require(cycle25_summary["supplementary_2026_outcomes_read"] is False,
        "2026 was already read during Cycle25 evaluation")
require(cycle25_summary["policy_reselection_performed"] is False,
        "Unexpected Cycle25 reselection")

cuts=pd.read_csv(CUTS)
mond=pd.read_csv(MOND_THRESHOLDS)

q={}
for branch in ["sharp","aia","sharp_aia"]:
    hit=mond.loc[(mond["branch"]==branch)&np.isclose(mond["alpha"].astype(float),PRIMARY_ALPHA)]
    require(len(hit)==1,f"Missing Mondrian qhat: {branch}")
    q[branch]=(float(hit.iloc[0]["qhat_class0"]),float(hit.iloc[0]["qhat_class1"]))

def cutoff(metric,pct):
    hit=cuts.loc[(cuts["metric"]==metric)&np.isclose(cuts["percentile"].astype(float),pct)]
    require(len(hit)==1,f"Missing frozen cutoff {metric} q{pct}")
    return float(hit.iloc[0]["cutoff"])

# Read only the original conformal-calibration role to freeze v3 state logic.
master=pd.read_csv(MASTER,low_memory=False)
sharp=pd.read_csv(SHARP,low_memory=False)
a=master.loc[master["role"].astype(str)==ROLE].copy()
s=sharp.loc[sharp["role"].astype(str)==ROLE].copy()

require(len(a)==EXPECTED_ROWS and len(s)==EXPECTED_ROWS,"Calibration support changed")
a["forecast_case_id"]=a["forecast_case_id"].astype(str)
s["forecast_case_id"]=s["forecast_case_id"].astype(str)
require(a["forecast_case_id"].tolist()==s["forecast_case_id"].tolist(),
        "Calibration ID/order mismatch")

aia_seed=a[[f"probability_seed_{z}" for z in SEEDS]].astype(float).to_numpy()
sharp_seed=s[[f"probability_seed_{z}" for z in SEEDS]].astype(float).to_numpy()
aia_p=pd.to_numeric(a["probability"],errors="coerce").to_numpy(float)
sharp_p=pd.to_numeric(s["probability_gru_mean"],errors="coerce").to_numpy(float)
fusion_seed=0.5*sharp_seed+0.5*aia_seed
fusion_p=0.5*sharp_p+0.5*aia_p

aia_status=a["input_status"].astype(str).to_numpy() if "input_status" in a.columns else np.array(["ok"]*len(a))
aia_ok=(aia_status=="ok")&np.isfinite(aia_p)&np.isfinite(aia_seed).all(axis=1)
sharp_ok=np.isfinite(sharp_p)&np.isfinite(sharp_seed).all(axis=1)
fusion_ok=aia_ok&sharp_ok&np.isfinite(fusion_p)

signals=pd.DataFrame({
    "forecast_case_id":a["forecast_case_id"].to_numpy(),
    "issue_utc":a["issue_utc"].to_numpy() if "issue_utc" in a.columns else pd.NA,
    "aia_input_status":aia_status,
    "sharp_input_status":np.where(sharp_ok,"ok","unavailable"),
    "sharp_probability":sharp_p,
    "aia_probability":aia_p,
    "fusion_probability":fusion_p,
    "sharp_seed_std":np.nanstd(sharp_seed,axis=1,ddof=1),
    "aia_seed_std":np.nanstd(aia_seed,axis=1,ddof=1),
    "fusion_seed_std":np.nanstd(fusion_seed,axis=1,ddof=1),
    "sharp_aia_probability_gap":np.abs(sharp_p-aia_p),
    "fusion_entropy":entropy_binary(fusion_p),
})

signals["sharp_mondrian_state"]=mondrian_state(sharp_p,*q["sharp"])
signals["aia_mondrian_state"]=mondrian_state(aia_p,*q["aia"])
signals["fusion_mondrian_state"]=mondrian_state(fusion_p,*q["sharp_aia"])

policies=[]
state_counts={}

for pct in PERCENTILES:
    fs=cutoff("fusion_seed_std",pct)
    gap=cutoff("sharp_aia_probability_gap",pct)
    ent=cutoff("fusion_entropy",pct)

    fusion_single=np.isin(
        signals["fusion_mondrian_state"].to_numpy(),
        ["singleton_no_flare","singleton_flare"]
    )

    # NORMAL remains conservative multimodal issue.
    normal=(
        fusion_ok &
        fusion_single &
        (signals["fusion_seed_std"].to_numpy()<=fs) &
        (signals["sharp_aia_probability_gap"].to_numpy()<=gap) &
        (signals["fusion_entropy"].to_numpy()<=ent)
    )

    # v3 redesign: SHARP availability is sufficient for fallback.
    # SHARP conformal/seed diagnostics are recorded but no longer block issuance.
    degraded=(~normal)&sharp_ok
    abstain=~(normal|degraded)

    require(np.all(normal.astype(int)+degraded.astype(int)+abstain.astype(int)==1),
            f"Non-exclusive v3 states q{pct}")

    pid=f"fallback_v3_q{int(pct*100)}"
    signals[f"{pid}_state"]=np.where(normal,"NORMAL",np.where(degraded,"DEGRADED","ABSTAIN"))

    state_counts[pid]={
        "NORMAL":int(normal.sum()),
        "DEGRADED":int(degraded.sum()),
        "ABSTAIN":int(abstain.sum()),
        "availability_rate":float((normal|degraded).mean()),
    }

    policies.append({
        "policy_id":pid,
        "cutoff_percentile":pct,
        "normal_rule":{
            "all":[
                "fusion technically available",
                "fusion Mondrian state is singleton",
                f"fusion_seed_std <= {fs:.17g}",
                f"sharp_aia_probability_gap <= {gap:.17g}",
                f"fusion_entropy <= {ent:.17g}",
            ],
            "forecast_source":"fixed 0.5/0.5 SHARP+AIA fusion",
        },
        "degraded_rule":{
            "all":[
                "NORMAL rule is false",
                "SHARP prediction is technically available"
            ],
            "forecast_source":"SHARP fallback",
            "note":"SHARP Mondrian state and seed spread are advisory diagnostics only; they do not block fallback issuance in v3."
        },
        "abstain_rule":"SHARP prediction is technically unavailable.",
    })

family={
    "status":"FALLBACK_FIRST_V3_POLICY_FAMILY_FROZEN",
    "scientific_acceptance":False,
    "primary_policy_id":"fallback_v3_q90",
    "sensitivity_policy_ids":["fallback_v3_q80","fallback_v3_q95"],
    "development_history":{
        "v1":"Marginal conformal + abstention failed rare-flare retention.",
        "v2":"Mondrian repaired calibration-class balance but still abstained disproportionately on Cycle25 flares.",
        "v3":"Redesigns uncertainty handling: uncertainty routes to SHARP fallback whenever SHARP is technically available; abstention is reserved for true fallback unavailability."
    },
    "cycle25_used_as_development_diagnostic":True,
    "cycle25_used_for_numeric_threshold_tuning":False,
    "supplementary_2026_used_for_design_or_tuning":False,
    "numeric_cutoffs_reused_unchanged":True,
    "candidates":policies,
    "calibration_block_state_counts":state_counts,
}

family_path=OUT/"policy_family_v3.json"
states_path=OUT/"calibration_policy_states_v3.csv.gz"
protocol_path=OUT/"protocol.json"

family_path.write_text(json.dumps(family,indent=2)+"\n")
signals.to_csv(states_path,index=False,compression={"method":"gzip","mtime":0})

protocol={
    "status":"FALLBACK_FIRST_V3_OPERATIONAL_POLICY_FROZEN",
    "scientific_acceptance":False,
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "horizon_hours":72,
    "primary_policy_id":"fallback_v3_q90",
    "sensitivity_policy_ids":["fallback_v3_q80","fallback_v3_q95"],
    "cycle25_seen_before_v3_design":True,
    "cycle25_role_after_v3":"development_diagnostic_only",
    "cycle25_used_for_numeric_threshold_tuning":False,
    "supplementary_2026_outcomes_read":False,
    "supplementary_2026_role":"clean_future_evaluation",
    "abstention_semantics":"ABSTAIN only when SHARP fallback is technically unavailable.",
    "normal_semantics":"Trusted multimodal fusion issue.",
    "degraded_semantics":"Issue SHARP fallback whenever fusion is not NORMAL but SHARP is technically available.",
    "source_sha256":{
        "master_predictions":MASTER_SHA,
        "sharp_predictions":SHARP_SHA,
        "mondrian_protocol":sha256(MOND_PROTOCOL),
        "mondrian_thresholds":sha256(MOND_THRESHOLDS),
        "candidate_cutoffs":sha256(CUTS),
        "alarm_thresholds":sha256(ALARM_THRESHOLDS),
        "v2_policy_protocol":sha256(V2_PROTOCOL),
        "cycle25_v2_summary":sha256(CYCLE25_SUMMARY),
    },
    "output_sha256":{
        family_path.name:sha256(family_path),
        states_path.name:sha256(states_path),
    },
    "next_stage":"Evaluate frozen fallback_v3_q90 once on supplementary_2026, with q80/q95 sensitivities, without reselection."
}
protocol_path.write_text(json.dumps(protocol,indent=2)+"\n")

print("===== FALLBACK-FIRST V3 POLICY FREEZE COMPLETE =====")
print("rows:",EXPECTED_ROWS)
print("Cycle25 used as development diagnostic: True")
print("Cycle25 used for numeric threshold tuning: False")
print("supplementary 2026 outcomes read: False")
print("primary policy: fallback_v3_q90")
print("\nCalibration-block state counts:")
print(json.dumps(state_counts,indent=2))
print("\nProtocol:")
print(json.dumps(protocol,indent=2))
print("\nSaved:",protocol_path)
print("Saved:",family_path)
print("Saved:",states_path)


===== FALLBACK-FIRST V3 POLICY FREEZE COMPLETE =====
rows: 4168
Cycle25 used as development diagnostic: True
Cycle25 used for numeric threshold tuning: False
supplementary 2026 outcomes read: False
primary policy: fallback_v3_q90

Calibration-block state counts:
{
  "fallback_v3_q80": {
    "NORMAL": 2819,
    "DEGRADED": 1349,
    "ABSTAIN": 0,
    "availability_rate": 1.0
  },
  "fallback_v3_q90": {
    "NORMAL": 3064,
    "DEGRADED": 1104,
    "ABSTAIN": 0,
    "availability_rate": 1.0
  },
  "fallback_v3_q95": {
    "NORMAL": 3242,
    "DEGRADED": 926,
    "ABSTAIN": 0,
    "availability_rate": 1.0
  }
}

Protocol:
{
  "status": "FALLBACK_FIRST_V3_OPERATIONAL_POLICY_FROZEN",
  "scientific_acceptance": false,
  "created_utc": "2026-10-05T18:04:45.930297+00:00",
  "horizon_hours": 72,
  "primary_policy_id": "fallback_v3_q90",
  "sensitivity_policy_ids": [
    "fallback_v3_q80",
    "fallback_v3_q95"
  ],
  "cycle25_seen_before_v3_design": true,
  "cycle25_role_after_v3": "development